In [0]:
%sql
CREATE OR REPLACE TABLE workspace.us_electricity.silver_fred_observations AS

SELECT
    series_id,

    TRY_CAST(observation_date_raw AS DATE) AS observation_date,

    CASE
        WHEN value_raw IS NULL
          OR TRIM(value_raw) = ''
          OR TRIM(value_raw) = '.'
        THEN NULL

        ELSE TRY_CAST(value_raw AS DOUBLE)
    END AS value,

    CASE
        WHEN TRY_CAST(observation_date_raw AS DATE) IS NULL
            THEN FALSE

        WHEN value_raw IS NULL
          OR TRIM(value_raw) = ''
          OR TRIM(value_raw) = '.'
            THEN FALSE

        WHEN TRY_CAST(value_raw AS DOUBLE) IS NULL
            THEN FALSE

        ELSE TRUE
    END AS is_valid,

    CASE
        WHEN TRY_CAST(observation_date_raw AS DATE) IS NULL
            THEN 'INVALID_DATE'

        WHEN value_raw IS NULL
          OR TRIM(value_raw) = ''
          OR TRIM(value_raw) = '.'
            THEN 'MISSING_VALUE'

        WHEN TRY_CAST(value_raw AS DOUBLE) IS NULL
            THEN 'INVALID_VALUE'

        ELSE 'VALID'
    END AS quality_status,

    source_system,

    current_timestamp() AS processed_at

FROM workspace.us_electricity.raw_fred_observations;